In [2]:
import pygmt
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
#print(pygmt.show_versions())  # incluye GMT y Ghostscript

In [35]:
## variables 
minlon, maxlon = -87, -82
minlat, maxlat = 7.0, 12.1
topo_data = '@earth_relief_30s' #arcosecond global relief SRTM
events = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/processing/bigger_6.txt" # events from Costa Rica
stations = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/CR_GNSS/lista.txt"

# magscale = "mag.dat"
nazca = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/nazca.dat"
cari = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/caribe.dat"
coco = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/coco2.dat"

In [ ]:
# import earthquakes
lons, lats, depth, date, mag = np.genfromtxt(events, usecols=(12, 11, 13, 0, 10), unpack=True,)

# secure the arrays
lons = np.asarray(lons, float)
lats = np.asarray(lats, float)
depth = np.asarray(depth, float)
mag   = np.asarray(mag,   float)

# Global mask to filter out invalid data
mask = np.isfinite(lons) & np.isfinite(lats) & np.isfinite(depth) & np.isfinite(mag)
lons_c, lats_c, depth_c, mag_c = lons[mask], lats[mask], depth[mask], mag[mask]
if lons_c.size == 0:
    raise ValueError("THere are no valid events.")

# Secure CPT range (avoid dmin==dmax and NaN)s
dmin = float(np.nanmin(depth_c))
dmax = float(np.nanmax(depth_c))
if dmin == dmax:
    eps = 1e-6 if dmin == 0 else abs(dmin)*1e-6
    dmin -= eps; dmax += eps

# (opcional) create a reasonable step
step = max((dmax - dmin)/20.0, 1e-3)


In [ ]:
# import plates
lonaz,latnaz = np.genfromtxt(nazca, usecols=(1, 0), unpack=True,)
lonca,latca = np.genfromtxt(cari, usecols=(1, 0), unpack=True,)
lonco,latco = np.genfromtxt(coco, usecols=(1, 0), unpack=True,)

#import stations
xsta, ysta,= np.genfromtxt(stations, usecols=(2, 1, ), unpack=True,)
df1 = pd.read_csv(stations,delim_whitespace=True,
                names=['name','latx','lony','z'])
name = np.asarray(df1.name)

/var/folders/fm/zxb00jfj0zdcxft7sfv1gps00000gn/T/ipykernel_16959/1695834467.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df1 = pd.read_csv(stations,delim_whitespace=True,


In [39]:
# Start plotting
fig = pygmt.Figure()
### Main figure
fig.shift_origin(yshift="3c", )
with fig.subplot(nrows=1, ncols=1, figsize=("15c", "15c"), ):
    fig.coast(
        region=[minlon, maxlon, minlat, maxlat],
        shorelines=True,
        borders=["1/0.5,black"],
        frame=["af", "WSen"],
    )
    # topography
    fig.grdimage(
        grid=topo_data,
        region=[minlon, maxlon, minlat, maxlat],
        cmap="gray",
        shading=True,
        transparency=30,
    )
    # contour
    fig.grdcontour(
        grid=topo_data,
        levels=500,
        pen=0.3,
    )
    fig.coast(map_scale=["-84/7.95/20/150+lKilometers/w0.5c", ], Tf="-82.67/11.5/1.6c/::",  shorelines="0.5", borders="1/1p")
    # plot plates
    fig.plot(x=lonaz, y=latnaz, pen="1p,darkred,-", )
    fig.plot(x=lonca, y=latca, pen="1p,darkred,-", )
    fig.plot(x=lonco, y=latco, pen="1p,darkred,-", )

#     # color pallets
    pygmt.makecpt(cmap='viridis', series=[dmin, dmax, step], reverse=True)


    # plot earthquakes
    fig.plot(
        x=lons_c, y=lats_c,
        fill=depth_c, cmap=True,
        size=0.077 * (1.3 ** mag_c),
        style="cc",
        pen="0.5,black",
        transparency=10,
        #label="Earthquakes bigger than 6"
    )

    #plot the stations
    fig.plot(
        x=xsta, y=ysta,
        style="t0.1s",
        pen="0.5,black",
        fill="red",
        #label="Stations"
    )
    # name stations
    for i, txt in enumerate(name):
        fig.text(x=xsta[i]+0.05, y=ysta[i]-0.05, text=txt, font="5p,Helvetica-Bold", fill="white", )


fig.savefig("/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/eathquakes.png", crop=True, dpi=700)


plot [WARNING]: Length <unit> s not supported - revert to default unit [cm]
